In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.decomposition import PCA
from xgboost import XGBClassifier

In [2]:
train = pd.read_parquet(r"C:\Users\lavan\Downloads\train_data.parquet")
user_behavior = pd.read_parquet(r"C:\Users\lavan\Downloads\add_event.parquet")
test = pd.read_parquet(r"C:\Users\lavan\Downloads\train_data.parquet")

In [3]:
train.head(2)


,id1,id2,id3,id4,id5,y,f1,f2,f3,f4,...,f357,f358,f359,f360,f361,f362,f363,f364,f365,f366
0,1366776_189706075_16-23_2023-11-02 22:22:00.042,1366776,189706075,2023-11-02 22:22:00.042,2023-11-02,0,1.0,None,None,None,...,None,-9999.0,0.0,None,28.0,0.0,0.0,337.0,0.0,0.0
1,1366776_89227_16-23_2023-11-01 23:51:24.999,1366776,89227,2023-11-01 23:51:24.999,2023-11-01,0,1.0,None,None,None,...,None,None,0.0,None,87.0,0.0,0.0,1010.0,2.0,0.0019801980198019


In [4]:
test.head(2)

,id1,id2,id3,id4,id5,y,f1,f2,f3,f4,...,f357,f358,f359,f360,f361,f362,f363,f364,f365,f366
0,1366776_189706075_16-23_2023-11-02 22:22:00.042,1366776,189706075,2023-11-02 22:22:00.042,2023-11-02,0,1.0,None,None,None,...,None,-9999.0,0.0,None,28.0,0.0,0.0,337.0,0.0,0.0
1,1366776_89227_16-23_2023-11-01 23:51:24.999,1366776,89227,2023-11-01 23:51:24.999,2023-11-01,0,1.0,None,None,None,...,None,None,0.0,None,87.0,0.0,0.0,1010.0,2.0,0.0019801980198019


In [5]:
user_behavior.head(2)

,id2,id3,id6,id4,id7
0,2431360,618619,Tiles,2023-10-22 08:08:17.768,None
1,2431360,363153,Tiles,2023-10-22 08:08:18.921,None


In [6]:
# Identify usable columns present in both datasets
usable_cols = [col for col in ['id2', 'id3', 'id4', 'id6', 'id7'] if col in user_behavior.columns and col in train.columns and col in test.columns]

# Feature Engineering on user_behavior
user_behavior['event_count'] = 1
user_agg = user_behavior.groupby(usable_cols).agg({
    'event_count': 'count'
}).reset_index()

if 'event_type' in user_behavior.columns:
    unique_event_counts = user_behavior.groupby(usable_cols)['event_type'].nunique().reset_index(name='unique_event_types')
    user_agg = user_agg.merge(unique_event_counts, on=usable_cols, how='left')
else:
    user_agg['unique_event_types'] = 0

In [7]:
X = train.drop(columns=['y'])
y = train['y']
X_test = test.drop(columns=['y'])
y_test = test['y']

In [8]:
X_test.shape

(770164, 371)

In [9]:
numeric_features = [col for col in ['event_count', 'unique_event_types', 'click_count'] if col in X.columns]
categorical_features = [col for col in ['most_common_event_type'] if col in X.columns]

In [10]:
y = y.astype(int)

In [11]:
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()


In [12]:
numeric_cols = [col for col in numeric_cols if X[col].notna().any()]

# Filter out categorical columns with all missing values
categorical_cols = [col for col in categorical_cols if X[col].notna().any()]


In [13]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


In [14]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [16]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        use_label_encoder=False,
        eval_metric='logloss',
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])


In [17]:
model.fit(X, y)


C:\Users\lavan\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [22:57:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
y_test_proba = model.predict_proba(X_test)[:, 1]

In [21]:
submission = pd.DataFrame({
    'id1': test['id1'],
    'id2': test['id2'],
    'id3': test['id3'],
    'id5': test['id5'],
    'pred': y_test_proba
})

submission.to_csv('submission.csv', index=False)
print("Submission file created from train/test split.")

Submission file created from train/test split.


In [ ]:
# this is in case everything goes down 
drop_cols = ['f20', 'f21', 'f34', 'f37', 'f80', 'f84', 'f112',
             'f120', 'f122', 'f135', 'f136', 'f189', 'f360', 'f377', 'id11']
train.drop(columns=drop_cols, inplace=True, errors='ignore')

In [ ]:
sparse_cols = train.columns[train.isnull().mean() > 0.99].tolist()
train.drop(columns=sparse_cols, inplace=True)

In [ ]:
target = 'y'
id_cols = ['customer_id', 'offer_id', 'id1', 'id2', 'id5', 'id8', 'id9', 'id10', 'id12', 'id13']
feature_cols = [col for col in train.columns if col not in id_cols + [target]]

In [ ]:
numeric_cols = train[feature_cols].select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = train[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()


In [ ]:
train_sample = train.sample(frac=0.599483, random_state=42)
X = train_sample[numeric_cols + categorical_cols]
y = train_sample[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)


In [ ]:
y_train = y_train.astype(int)
y_test = y_test.astype(int)
y_train = y_train.iloc[:-59]
X_train = X_train.iloc[:-59]

In [ ]:
X_test.shape

In [ ]:
def rare_category_grouper(X, threshold=0.01):
    if isinstance(X, np.ndarray):
        X = pd.DataFrame(X, columns=categorical_cols)
    X_copy = X.copy()
    for col in X_copy.columns:
        freq = X_copy[col].value_counts(normalize=True)
        rare_labels = freq[freq < threshold].index
        X_copy[col] = X_copy[col].apply(lambda x: 'Other' if x in rare_labels else x)
    return X_copy


In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])


In [ ]:
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('rare_label', FunctionTransformer(rare_category_grouper, validate=False)),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])


In [ ]:
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])


In [ ]:
neg, pos = np.bincount(y_train)
imbalance_ratio = neg / pos
xgb_model = XGBClassifier(
    eval_metric='logloss',
    n_estimators=300,
    max_depth=3,
    subsample=0.7,
    colsample_bytree=0.7,
    learning_rate=0.05,
    scale_pos_weight=imbalance_ratio,
    reg_lambda=10.0,
    reg_alpha=2.0,
    random_state=42,
    use_label_encoder=False
)

In [ ]:
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', xgb_model)
])


In [ ]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

In [ ]:
submission = train_sample.loc[X_test.index, ['id1','id2', 'id3', 'id5']].copy()
submission['pred'] = y_proba
submission.to_csv("r2_submission_file_4.csv", index=False)
print("Submission file created from train/test split.")